## Chunking

Detta är en överkomplicerad chunking metod som jag först trodde skulle få bra resultat. Sen visade det sig att lesson_id inte följdes nummeriskt i ordning.
Därför blev den värdelös.

In [11]:
import os
import json
from google import genai
from dotenv import load_dotenv
from google.genai import types
import numpy as np
import polars as pl

In [12]:
def load_records_from_cleaned(folder_path):
    records = []

    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".json"):
                full_path = os.path.join(root, file)

                with open(full_path, "r", encoding="utf-8") as f:
                    data = json.load(f)

                    records.append({
                        "course_name": data.get("course_name"),
                        "chapter_title": data.get("chapter_title"),
                        "lesson_id": data.get("lesson_id"),
                        "lesson_title": data.get("lesson_title"),
                        "text": data.get("text"),
                        "file_path": full_path
                    })

    return records

In [13]:
folder = r"C:\YA BI analyst\Kurser\BI25M AI & IoT\Kunskapskontroll 1\AI-IoT-Kunskapskrav-1\data\cleaned"

records = load_records_from_cleaned(folder)

print(f"Antal records: {len(records)}")
print(records[0])  # sanity check

Antal records: 349
{'course_name': 'Ansvariga- Vem ska jag kontakta?', 'chapter_title': 'Vem ska jag kontakta?', 'lesson_id': '31459054', 'lesson_title': 'Båtskada', 'text': 'Olika skeppare har olika erfarenhet av olika båtar, var därför inte rädd för att fråga föregående skeppare om saker du undrar över. Frågor om båten bör du ställa redan på bytesdagen och självklart kan du alltid höra av dig under veckan om du har ytterligare frågor. Vet du inte vem som varit på båten tidigare eller har föregående skepparen inte svar på dina frågor? Då skriver du i skeppargruppen eller en av dina mentorer.\nSkeppargruppen skall kunna användas för att fråga om eventuella strul, väder eller generella frågor. Om du mot förmodan inte får ett vettigt svar är det bäst att kontakta din\nplatsansvarig.\nHar du en skada efter till exempel en tilläggning eller liknande ska du kontakta din\nplatsansvarig samt ansvarig tekniker för din båt!', 'file_path': 'C:\\YA BI analyst\\Kurser\\BI25M AI & IoT\\Kunskapskont

In [14]:
import pandas as pd
import re

# Hjälpfunktioner
# funktioner som används inom funktionerna
# ==========================================================================================

def is_empty_text(text):
    """
    Returnerar True om texten är tom eller bara innehåller whitespace.
    """
    return not text or not str(text).strip()


def split_into_paragraphs(text: str):
    """
    Delar text i stycken baserat på tomrader.
    """
    if is_empty_text(text):
        return []

    paragraphs = re.split(r"\n\s*\n+", text.strip())
    return [p.strip() for p in paragraphs if p.strip()]

# Behövs för att få med metadata och visa vilken lektion texten kommer ifrån, är användbart vid retrieval och felsökning
def format_lesson_block(lesson_title: str, text: str):
    """
    Skapar en rubrik med lektionsnamn
    när lektioner slås ihop med rubrik + text.
    """
    return f"Lektion: {lesson_title}\n{text}".strip()

# Behövs för att få med metadata och visa vilken kurs och lektion texten kommer ifrån, är användbart vid retrieval och felsökning
def format_chunk_text(course_name: str, chapter_title: str, body: str):
    """
    Skapar full chunktext med kurs + kapitel + innehåll.
    """
    return (
        f"Kurs: {course_name}\n"
        f"Kapitel: {chapter_title}\n\n"
        f"{body}"
    ).strip()


def create_chunk(course_name, chapter_title, body, lesson_titles, lesson_ids, source_files):
    """
    Skapar ett standardiserat chunk-objekt.
    """
    chunk_text = format_chunk_text(course_name, chapter_title, body)

    # Egentligen behövs bara "chunk_text" för att göra chunksen, övrig metadata returns är mest med för att kunna granska chunksen och felsöka innan vi skickar in det för embedding,
    # men också vid evealueringen. Man skulle också kunna använda metadata i chattbotten för att sortera men det är inget jag planerat för hittls.
    return {
        "course_name": course_name,
        "chapter_title": chapter_title,
        "lesson_titles": lesson_titles,
        "lesson_ids": lesson_ids,
        "lesson_count": len(lesson_ids),
        "text": chunk_text,
        "char_count": len(chunk_text),
        "word_count": len(chunk_text.split()),
        "source_files": source_files,
    }


def merge_small_last_text_chunk(chunks, target_chars, max_chars):
    """
    Slår ihop sista textchunken bakåt om den är väldigt liten.
    """
    #Kollar om det inte finns någon chunk innan
    if len(chunks) < 2:
        return chunks
    
    last = chunks[-1]
    prev = chunks[-2]

    # Ta bort header från sista chunken
    if "\n" in last:
        last_body = last.split("\n", 1)[1]
    else:
        last_body = last

    merged = prev + "\n\n" + last_body

    # Gör bara merge om texten inte blir större än max_chars
    if len(merged) <= max_chars:
        chunks[-2] = merged.strip()
        chunks.pop()

    return chunks


def split_large_lesson_by_linebreaks(lesson_title, text, target_chars=1800, max_chars=2200):
    """
    Fallback: delar text på radbrytning om inga bra stycken finns.
    Har också en inbyggd ValueError om chunken skulle råka bli tom

    """
    header = f"Lektion: {lesson_title}\n"
    text = text.strip()

    chunks = []
    start = 0
    n = len(text)

    while start < n:
        end = min(start + target_chars, n)

        # Försöker chunka vid närmsta radbrytning bakåt istället för att chunka hårt
        if end < n:
            break_pos = text.rfind("\n", start, end)
            if break_pos > start:
                end = break_pos
                

        # Enkel säkerhet så vi inte fastnar i edge cases där inget radslut hittas inom target_chars. 
        if end <= start:
            raise ValueError(
            f"Chunking fastnade i split_large_lesson_by_chars: "
            f"lesson_title={lesson_title!r}, start={start}, end={end}"
        )
        
        chunk_body = text[start:end].strip()

        if chunk_body:
            chunks.append((header + chunk_body).strip())

        start = end

    return merge_small_last_text_chunk(chunks, target_chars, max_chars)

# ==========================================================================================

def split_large_lesson(lesson_title, text, target_chars=1800, max_chars=2200):
    """
    Delar en stor lesson i mindre delar.

    Försöker i första hand bygga chunks av hela stycken.
    Vi siktar på target_chars, men tillåter större chunks upp till
    max_chars om det gör att vi kan behålla ett helt stycke.
    """
    if is_empty_text(text):
        return []

    header = f"Lektion: {lesson_title}\n"
    paragraphs = split_into_paragraphs(text)

    # Om inga riktiga stycken finns använder vi fallback
    if len(paragraphs) <= 1:
        return split_large_lesson_by_linebreaks(lesson_title, text, target_chars, max_chars)

    chunks = []
    current = ""

    for para in paragraphs:
        if not current:
            current = para
            continue

        candidate = current + "\n\n" + para

        # Bygg gärna upp chunken till ungefär target_chars
        if len(header + candidate) <= target_chars:
            current = candidate
            continue

        # Om nästa stycke gör chunken större än target men fortfarande
        # inom max_chars, behåll hela stycket istället för att bryta tidigt
        if len(header + candidate) <= max_chars:
            current = candidate
            continue

        # Annars sparar vi nuvarande chunk och börjar en ny
        chunks.append((header + current).strip())
        current = para

    if current:
        chunks.append((header + current).strip())

    return merge_small_last_text_chunk(chunks, target_chars, max_chars)


# Denna funktion las till i efterhand när jag insåg att det fanns vissa kurser använder kapitel på samma sätt som andra kurser använder lektioner
# Jag behövde därför en funktion som i efterhand slår ihop väldigt små chunks av kapitel till en chunk. 
# Här visade det sig väldigt bra att jag sparat metadatat hela vägen.
def merge_small_last_chapter_chunk(chapter_chunks, course_name, chapter_title, target_chars, max_chars):
    """
    Slår ihop sista chunk i kapitlet bakåt om den är väldigt liten.
    """
    # Kollar så att det finns två chunks i kapitlet
    if len(chapter_chunks) < 2:
        return chapter_chunks

    # Chunkar bara ihop lektionerna om de anses tillräckligt små.
    if chapter_chunks[-1]["char_count"] >= target_chars * 0.35:
        return chapter_chunks

    last = chapter_chunks[-1]
    prev = chapter_chunks[-2]

    # Plockar ut texten från föregående och nästa chunk
    prev_body = prev["text"].split("\n\n", 1)[1]
    last_body = last["text"].split("\n\n", 1)[1]

    merged_body = prev_body + "\n\n" + last_body
    merged_text = format_chunk_text(course_name, chapter_title, merged_body)

    # Lägger bara ihop texterna om de inte blir längre än max_chars
    # Slår ihop metadata så att de stämmer.
    if len(merged_text) <= max_chars:
        prev["text"] = merged_text
        prev["lesson_titles"].extend(last["lesson_titles"])
        prev["lesson_ids"].extend(last["lesson_ids"])
        prev["lesson_count"] += last["lesson_count"]
        prev["source_files"].extend(last["source_files"])
        prev["char_count"] = len(merged_text)
        prev["word_count"] = len(merged_text.split())
        # Tar bort den texten som slogs ihop med den andre, annars får vi chunk 1 = text 1 + text 2 och chunk 2 = text 2
        chapter_chunks.pop()

    return chapter_chunks


def build_lesson_chunks(records, target_chars=1800, max_chars=2200):
    """
    Huvudfunktion för att bygga chunks.

    Flöde:
    1. Filtrera bort tomma lessons
    2. Sortera data
    3. Gruppera per kurs + kapitel
    4. Chunka lessons
    5. Slå ihop små sista chunks
    """

    # Ta bort tomma lessons som vi vet att det finns 37st av på alla 11 kurser.
    # använder is_empty_text hjälpfunktionen
    records = [r for r in records if not is_empty_text(r.get("text"))]

    df = pd.DataFrame(records)

    # Sorterar datat för att loopen ska fungera
    sort_cols = [col for col in ["course_name", "chapter_title", "lesson_id"] if col in df.columns]
    df = df.sort_values(sort_cols, na_position="last").reset_index(drop=True)

    # Skapar en lista för alla chunks
    chunks = []

    # Grupperar på kurser så att vi inte kan slå ihop texter från olika kurser.
    for (course_name, chapter_title), group in df.groupby(["course_name", "chapter_title"], dropna=False):
        group = group.reset_index(drop=True)
        chapter_chunks = []

        # While loop genom lessons i kapitlet
        i = 0
        while i < len(group):
            row = group.iloc[i]

            lesson_title = row.get("lesson_title", "")
            text = row.get("text", "")

            # Använder en format_lesson_block funktionen för att få med lektionen som rubrik
            lesson_block = format_lesson_block(lesson_title, text)

            # Splittar lektionen om den är för stor använder hjälpfunktionen split_large_lesson
            if len(lesson_block) > max_chars:
                split_blocks = split_large_lesson(lesson_title, text, target_chars, max_chars)

                #Skapar chunks för varje splitad del
                for block in split_blocks:
                    chapter_chunks.append(
                        create_chunk(
                            course_name,
                            chapter_title,
                            block,
                            [lesson_title],
                            [row.get("lesson_id")],
                            [row.get("file_path")]
                        )
                    )
                # Går vidare till nästa lektion
                i += 1
                continue

            # Är lektionen inte för stor så sparas den och går vidare
            current_text = lesson_block
            lesson_titles = [lesson_title]
            lesson_ids = [row.get("lesson_id")]
            source_files = [row.get("file_path")]

            # Tittar på nästa lektion
            j = i + 1

            # Försöker lägga till fler lektioner så länge det finns fler i kapitlet
            while j < len(group):
                if len(current_text) >= target_chars:
                    break
                
                # Hämtar nästa lektion
                next_row = group.iloc[j]
                next_title = next_row.get("lesson_title", "")
                next_text = next_row.get("text", "")
                next_block = format_lesson_block(next_title, next_text)

                # Kollar om nästa lektion är längre än max_chars
                if len(next_block) > max_chars:
                    break
                    
                candidate = current_text + "\n\n" + next_block

                # Lägger ihop lektionerna och uppdaterar metadata om texterna får plats inom max_chars
                if len(candidate) <= max_chars:
                    current_text = candidate
                    lesson_titles.append(next_title)
                    lesson_ids.append(next_row.get("lesson_id"))
                    source_files.append(next_row.get("file_path"))
                    j += 1
                else:
                    break

            # Skapar ett chunk-objekt av allt som samlats i current_text
            chapter_chunks.append(
                create_chunk(
                    course_name,
                    chapter_title,
                    current_text,
                    lesson_titles,
                    lesson_ids,
                    source_files
                )
            )
            # Går till nästa lektion
            i = j

        # Här slår vi ihop väldigt små kapitel som förmodligen borde varit lektioner
        # Vi använder hjälpfunktionen merge_small_last_chapter_chunk
        chapter_chunks = merge_small_last_chapter_chunk(
            chapter_chunks, course_name, chapter_title, target_chars, max_chars
        )

        #Lägger till kapitel chunksen
        chunks.extend(chapter_chunks)

    return chunks

In [15]:
lesson_chunks = build_lesson_chunks(records, target_chars=1700, max_chars=2200)

print(f"Antal lesson_chunks skapade: {len(lesson_chunks)}")

chunk_df = pd.DataFrame(lesson_chunks).sort_values("char_count", ascending=False).reset_index(drop=True)
"""
print("\nStörsta chunks:\n")
print(
    chunk_df[
        ["course_name", "chapter_title", "lesson_count", "char_count", "word_count"]
    ].head(50).to_string(index=False)
)
"""
print("\nMinsta chunks:\n")
print(
    chunk_df.sort_values("char_count", ascending=True)[
        ["course_name", "chapter_title", "lesson_count", "char_count", "word_count"]
    ].head(30).to_string(index=False)
)

Antal lesson_chunks skapade: 282

Minsta chunks:

                     course_name                  chapter_title  lesson_count  char_count  word_count
                      Våra båtar                     Våra båtar             1         205          34
                      Riktlinjer    Uppträdande & kommunikation             1         364          48
                    Båthantering                    Tilläggning             1         387          63
    Genomförande av seglingsresa  Säkerhetsgenomgång med gäster             1         434          66
     More Sailing - vilka är vi?          Välkommen till gänget             1         458          90
                     Kundnöjdhet        Inte så nöjda kunder :(             1         470          81
                    Mathantering                   Specialkost              1         530          93
           Destination: Kroatien                        Korcula             2         553          93
                      Våra båtar

In [7]:
def second_pass_merge(chunks, max_chars):
    """
    Gör ett extra merge-pass för att slå ihop chunks som råkade bli små.
    Körs efter all chunking.
    """

    if not chunks:
        return chunks

    merged_chunks = []
    i = 0

    while i < len(chunks):
        current = chunks[i]

        # Försök mergea med nästa
        if i < len(chunks) - 1:
            nxt = chunks[i + 1]

            # Samma kapitel?
            if (
                current["course_name"] == nxt["course_name"]
                and current["chapter_title"] == nxt["chapter_title"]
            ):
                combined_text = current["text"] + "\n\n" + nxt["text"].split("\n\n", 1)[1]

                if len(combined_text) <= max_chars:
                    merged_chunks.append({
                        **current,
                        "text": combined_text,
                        "lesson_titles": current["lesson_titles"] + nxt["lesson_titles"],
                        "lesson_ids": current["lesson_ids"] + nxt["lesson_ids"],
                        "lesson_count": current["lesson_count"] + nxt["lesson_count"],
                        "source_files": current["source_files"] + nxt["source_files"],
                        "char_count": len(combined_text),
                        "word_count": len(combined_text.split())
                    })
                    i += 2
                    continue

        merged_chunks.append(current)
        i += 1

    return merged_chunks

In [8]:
chunks = second_pass_merge(lesson_chunks, max_chars=2200)

In [9]:
print(f"Antal lesson_chunks skapade: {len(chunks)}")

chunk_df = pd.DataFrame(chunks).sort_values("char_count", ascending=False).reset_index(drop=True)
"""
print("\nStörsta chunks:\n")
print(
    chunk_df[
        ["course_name", "chapter_title", "lesson_count", "char_count", "word_count"]
    ].head(50).to_string(index=False)
)
"""
print("\nMinsta chunks:\n")
print(
    chunk_df.sort_values("char_count", ascending=True)[
        ["course_name", "chapter_title", "lesson_count", "char_count", "word_count"]
    ].head(30).to_string(index=False)
)

Antal lesson_chunks skapade: 273

Minsta chunks:

                     course_name                                                         chapter_title  lesson_count  char_count  word_count
                      Våra båtar                                                            Våra båtar             1         205          34
    Genomförande av seglingsresa                                         Säkerhetsgenomgång med gäster             1         434          66
     More Sailing - vilka är vi?                                                 Välkommen till gänget             1         458          90
                     Kundnöjdhet                                               Inte så nöjda kunder :(             1         470          81
                    Mathantering                                                          Specialkost              1         530          93
           Destination: Kroatien                                                               Korcula  

In [10]:
original_texts = {c["text"] for c in original_chunks}
new_texts = {c["text"] for c in chunks}

removed_texts = original_texts - new_texts

print(f"Antal borttagna chunks: {len(removed_texts)}")

Antal borttagna chunks: 18


In [11]:
for text in removed_texts:
    print("\n--- BORTAGEN CHUNK ---\n")
    print(text[:500])  # visar första 500 tecken


--- BORTAGEN CHUNK ---

Kurs: Destination: Kroatien
Kapitel: Vis

Lektion: Allmänt Vis
Under 1860-talet drabbades Europa av vinpesten, vinpesten är en liten insekt som sätter sig på vinrankorna och äter upp dess blad och rötter så att vinrankorna dör. Detta gjorde att nästan all vinodling i Europa försvann, dock klarade sig Vis mycket bra p.g.a sitt isolerade läge mitt ute i det Adriatiska havet. Eftersom Europas vinproduktion störtdök blev vin plötsligt en bristvara och de som kunde producera vin tjänade stora pengar.

--- BORTAGEN CHUNK ---

Kurs: Destination: Kroatien
Kapitel: Fastlandet Norr om Trogir 

Lektion: Primošten
Svart prick: Gömda stranden
Grön prick: Restaurang Agape - mycket bra och varierad mat, vi har tidigare ätit mycket här och har en bra relation till familjen. Nämner ni att ni kommer från More Sailing kommer göra allt för att ordna ett bord till er.
Grön/röd prick: Tomahawk steakhouse - bästa köttet i Primošten
Rosa prick: Hamnkontor, dusch och toalett
Blå prick:

In [4]:
import pandas as pd
import re


def split_into_paragraphs(text: str):
    """
    Delar text i stycken baserat på tomrader.
    Om inga tydliga tomrader finns returneras hela texten som ett stycke.
    """
    if not text or not text.strip():
        return []

    paragraphs = re.split(r"\n\s*\n+", text.strip())
    paragraphs = [p.strip() for p in paragraphs if p.strip()]
    return paragraphs


def split_large_lesson(lesson_title: str, text: str, target_chars=1800, max_chars=2200):
    """
    Delar en enskild stor lesson i mindre delar.

    Logik:
    - Försök först dela på stycken
    - Bygg upp delar mot target_chars
    - Tillåt upp till max_chars om det hjälper att undvika små restbitar
    - Om texten i praktiken bara är ett stort stycke, fallback till teckensplit

    Returnerar en lista av textblock där varje block redan innehåller lesson-rubriken.
    """
    header = f"Lektion: {lesson_title}\n"

    if not text or not text.strip():
        return []

    paragraphs = split_into_paragraphs(text)

    # Om inga riktiga stycken finns, eller bara ett enda jättestycke:
    if len(paragraphs) <= 1:
        return split_large_lesson_by_chars(lesson_title, text, target_chars=target_chars, max_chars=max_chars)

    chunks = []
    current = ""

    for para in paragraphs:
        if not current:
            current = para
            continue

        candidate = current + "\n\n" + para

        # Normalregel: fortsätt om vi håller oss under target
        if len(header + candidate) <= target_chars:
            current = candidate
            continue

        # Om current redan är tillräckligt stor, spara och börja ny
        if len(header + current) >= target_chars * 0.6:
            chunks.append((header + current).strip())
            current = para
            continue

        # Om current är lite liten kan vi tillåta upp till max
        if len(header + candidate) <= max_chars:
            current = candidate
        else:
            chunks.append((header + current).strip())
            current = para

    if current:
        chunks.append((header + current).strip())

    # Slå ihop sista liten restchunk bakåt om möjligt
    if len(chunks) >= 2 and len(chunks[-1]) < target_chars * 0.35:
        merged = chunks[-2] + "\n\n" + chunks[-1].split("\n", 1)[1]
        if len(merged) <= max_chars:
            chunks[-2] = merged.strip()
            chunks.pop()

    return chunks


def split_large_lesson_by_chars(lesson_title: str, text: str, target_chars=1800, max_chars=2200):
    """
    Fallback om en lesson inte går att dela vettigt på stycken.

    Delar texten i teckenblock som siktar på target_chars.
    Försöker bryta vid mellanslag.
    Slår ihop sista liten restbit bakåt om möjligt.
    """
    header = f"Lektion: {lesson_title}\n"

    text = text.strip()
    chunks = []
    start = 0
    n = len(text)

    while start < n:
        end = min(start + target_chars, n)

        if end < n:
            break_pos = text.rfind(" ", start, end)
            if break_pos > start:
                end = break_pos

        chunk_body = text[start:end].strip()

        if not chunk_body:
            end = min(start + target_chars, n)
            chunk_body = text[start:end].strip()

        if chunk_body:
            chunks.append((header + chunk_body).strip())

        start = end

    # Slå ihop sista liten restchunk bakåt om möjligt
    if len(chunks) >= 2 and len(chunks[-1]) < target_chars * 0.35:
        merged = chunks[-2] + " " + chunks[-1].split("\n", 1)[1]
        if len(merged) <= max_chars:
            chunks[-2] = merged.strip()
            chunks.pop()

    return chunks


def build_lesson_chunks(records, target_chars=1800, max_chars=2200):
    """
    Bygger chunks genom att utgå från lessons.

    Regler:
    - Börja med en lesson
    - Om lessonen är större än max_chars:
        -> splitta den först internt
    - Lägg annars till nästa lesson inom samma kapitel så länge:
        1. nuvarande chunk < target_chars
        2. total längd efter merge <= max_chars
    - Behåll lesson-rubriker i texten för semantik
    - Om sista chunk i kapitlet blir väldigt liten:
        -> försök slå ihop den bakåt
    """

    if not records:
        return []

    df = pd.DataFrame(records)

    sort_cols = [col for col in ["course_name", "chapter_title", "lesson_id", "lesson_title"] if col in df.columns]
    df = df.sort_values(sort_cols, na_position="last").reset_index(drop=True)

    chunks = []

    for (course_name, chapter_title), group in df.groupby(["course_name", "chapter_title"], dropna=False):
        group = group.reset_index(drop=True)

        chapter_chunks = []
        i = 0

        while i < len(group):
            row = group.iloc[i]

            lesson_title = row["lesson_title"] if pd.notna(row["lesson_title"]) else ""
            text = row["text"] if pd.notna(row["text"]) else ""

            if not text or not str(text).strip():
                i += 1
                continue

            block = f"Lektion: {lesson_title}\n{text}".strip()

            # --------------------------------------------------
            # FALL 1: lesson är större än max_chars
            # --------------------------------------------------
            if len(block) > max_chars:
                split_blocks = split_large_lesson(
                    lesson_title=lesson_title,
                    text=text,
                    target_chars=target_chars,
                    max_chars=max_chars
                )

                for split_block in split_blocks:
                    chunk_text = (
                        f"Kurs: {course_name}\n"
                        f"Kapitel: {chapter_title}\n\n"
                        f"{split_block}"
                    ).strip()

                    chapter_chunks.append({
                        "course_name": course_name,
                        "chapter_title": chapter_title,
                        "lesson_titles": [lesson_title],
                        "lesson_ids": [row.get("lesson_id")],
                        "lesson_count": 1,
                        "text": chunk_text,
                        "char_count": len(chunk_text),
                        "word_count": len(chunk_text.split()),
                        "source_files": [row.get("file_path")],
                    })

                i += 1
                continue

            # --------------------------------------------------
            # FALL 2: normal lesson -> mergea framåt som tidigare
            # --------------------------------------------------
            current_lessons = []
            current_source_files = []
            current_lesson_titles = []

            current_text = block

            current_lessons.append(row.get("lesson_id"))
            current_source_files.append(row.get("file_path"))
            current_lesson_titles.append(lesson_title)

            j = i + 1

            while j < len(group):
                if len(current_text) >= target_chars:
                    break

                next_row = group.iloc[j]
                next_lesson_title = next_row["lesson_title"] if pd.notna(next_row["lesson_title"]) else ""
                next_text = next_row["text"] if pd.notna(next_row["text"]) else ""

                if not next_text or not str(next_text).strip():
                    j += 1
                    continue

                next_block = f"Lektion: {next_lesson_title}\n{next_text}".strip()

                # Om nästa lesson ensam är för stor ska den hanteras separat i nästa varv
                if len(next_block) > max_chars:
                    break

                separator = "\n\n" + ("-" * 80) + "\n\n"
                candidate_text = current_text + separator + next_block

                if len(candidate_text) <= max_chars:
                    current_text = candidate_text
                    current_lessons.append(next_row.get("lesson_id"))
                    current_source_files.append(next_row.get("file_path"))
                    current_lesson_titles.append(next_lesson_title)
                    j += 1
                else:
                    break

            chunk_text = (
                f"Kurs: {course_name}\n"
                f"Kapitel: {chapter_title}\n\n"
                f"{current_text}"
            ).strip()

            chapter_chunks.append({
                "course_name": course_name,
                "chapter_title": chapter_title,
                "lesson_titles": current_lesson_titles,
                "lesson_ids": current_lessons,
                "lesson_count": len(current_lessons),
                "text": chunk_text,
                "char_count": len(chunk_text),
                "word_count": len(chunk_text.split()),
                "source_files": current_source_files,
            })

            i = j

        # --------------------------------------------------
        # Efterregel: om sista chunken i kapitlet är väldigt liten,
        # försök slå ihop den bakåt
        # --------------------------------------------------
        if len(chapter_chunks) >= 2 and chapter_chunks[-1]["char_count"] < target_chars * 0.35:
            last_chunk = chapter_chunks[-1]
            prev_chunk = chapter_chunks[-2]

            prev_body = prev_chunk["text"].split("\n\n", 1)[1]
            last_body = last_chunk["text"].split("\n\n", 1)[1]

            merged_text = (
                f"Kurs: {course_name}\n"
                f"Kapitel: {chapter_title}\n\n"
                f"{prev_body}\n\n{'-' * 80}\n\n{last_body}"
            ).strip()

            if len(merged_text) <= max_chars:
                prev_chunk["text"] = merged_text
                prev_chunk["lesson_titles"].extend(last_chunk["lesson_titles"])
                prev_chunk["lesson_ids"].extend(last_chunk["lesson_ids"])
                prev_chunk["lesson_count"] += last_chunk["lesson_count"]
                prev_chunk["source_files"].extend(last_chunk["source_files"])
                prev_chunk["char_count"] = len(merged_text)
                prev_chunk["word_count"] = len(merged_text.split())
                chapter_chunks.pop()

        chunks.extend(chapter_chunks)

    return chunks

In [5]:
lesson_chunks = build_lesson_chunks(records, target_chars=1800, max_chars=2200)

print(f"Antal lesson_chunks skapade: {len(lesson_chunks)}")

chunk_df = pd.DataFrame(lesson_chunks).sort_values("char_count", ascending=False).reset_index(drop=True)

print("\nStörsta chunks:\n")
print(
    chunk_df[
        ["course_name", "chapter_title", "lesson_count", "char_count", "word_count"]
    ].head(50).to_string(index=False)
)

print("\nMinsta chunks:\n")
print(
    chunk_df.sort_values("char_count", ascending=True)[
        ["course_name", "chapter_title", "lesson_count", "char_count", "word_count"]
    ].head(30).to_string(index=False)
)

Antal lesson_chunks skapade: 287

Största chunks:

                     course_name                                                         chapter_title  lesson_count  char_count  word_count
                    Mathantering Tidigare More Sailing recept (vissa är kvar i nya recepthäftet också)             3        2244         360
                    Mathantering                                                          Specialkost              2        2241         351
                    Båthantering                                           Felsökning av båt, skeppare             3        2238         302
                    Mathantering Tidigare More Sailing recept (vissa är kvar i nya recepthäftet också)             2        2235         389
           Destination: Kroatien                                                                  Brač             1        2221         403
                    Båthantering                                                           Tilläggning 

#### Tar bort en meningslös chunk manuellt

Här ser jag en chunk som jag misstänker är en värdelös intro chunk som inte mergeats av våran chunking logik. Man kan inte bygga en logik som fångar allt och texten är inte heller uppbyggt så att allt kan mergeas snyggt. Jag väljer därför att inspektera den för att se om jag hade rätt, vilket jag hade och därför tar jag sedan bort den.

In [6]:
df = pd.DataFrame(lesson_chunks)

target = df[
    (df["course_name"] == "Våra båtar") &
    (df["chapter_title"] == "Våra båtar")
]

print(target.iloc[0]["text"])

Kurs: Våra båtar
Kapitel: Våra båtar

Lektion: En översikt
Vi har många olika båtar i våra flottor i Kroatien & Grekland, så här följer en översikt av de vanligaste båttyperna vi använder för paketresorna.


In [7]:
lesson_chunks = [
    c for c in lesson_chunks
    if not (
        c["course_name"] == "Våra båtar"
        and c["chapter_title"] == "Våra båtar"
        and "översikt" in c["text"].lower()
    )
]

In [8]:
print([
    c for c in lesson_chunks
    if c["course_name"] == "Våra båtar"
    and c["chapter_title"] == "Våra båtar"
])

[]


Här kollar jag hur många dokument som blev chunkade

## Embeddings
Här testade jag först med gemini embedding men blev snabbt problem med request limits. Detta löste jag igenom att sätta endast köra 95 chunks i taget och sedan vänta 65 sekunder. Då kom jag under gratis versionens gränser och slapp köra det på en lokal modell.

--OBS SKRIV OM--
EmbeddingService skapades som en klass för att kapsla logiken kring embeddings och göra koden mer återanvändbar och strukturerad.

In [ ]:
from embedding_service import EmbeddingService
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

embedding_service = EmbeddingService(client, types)

In [14]:
embedded_docs = embedding_service.embed_documents_with_delay(
    lesson_chunks,
    batch_size=50,
    wait_seconds=65
)

Embedding 1/286


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0\nPlease retry in 32.940887086s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerUserPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-embedding-1.0', 'locati

In [ ]:
embedded_docs = embedding_service.embed_documents(lesson_chunks)

df = embedding_service.to_polars_df(embedded_docs)
print(df.head())

embedding_service.save_to_parquet(embedded_docs, "embeddings_v1.parquet")